# 04 · Choosing how to build the model

Every decision about the model is made in this notebook, using only data from before 1 Sep 2024,
the day the final test starts. If I tuned the model on the same months I test it on, it would
look better than it really is.

The practice runs work like the final test. On the first of April, May, June and July, the model
learns from what was known by then and is scored on the quotes sent that month. Quotes sent after
17 July are left out, because some of their replies hadn't arrived by 1 September.

Two questions, in order: which facts about a quote belong in the model, and how best to use
"days without a reply". The main yardstick is **log loss**, a standard score for predicted chances
that punishes confident mistakes (lower is better). When two options tie on it, I pick the one
whose chances hold up better group by group (H3).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))

import ablations
from quote_followup.config import load_settings
from quote_followup.data import load_quotes

pd.set_option("display.width", 160)
settings = load_settings(ROOT / "config.toml")
quotes = ablations.with_candidate_inputs(load_quotes(settings.data_dir))
print("practice runs start on:", ", ".join(f"{d:%d %b}" for d in ablations.ORIGINS),
      "| last quote used was sent on:", f"{ablations.BACKTEST_START - pd.Timedelta(days=settings.model.label_horizon_days):%d %b %Y}")

practice runs start on: 01 Apr, 01 May, 01 Jun, 01 Jul | last quote used was sent on: 18 Jul 2024


## Which facts about a quote matter?

Every fact tried here is known the moment the quote goes out. The two "history" facts are a
business's (or customer's) past win rate, using only replies already in, pulled towards the
overall rate when there's little history. Besides log loss, the table shows **AUC**: the chance
that a quote which said yes was rated higher than one that said no (0.5 is a coin flip, 1 is perfect).

In [2]:
send_time = ablations.send_time_ablations(quotes, settings)
display(send_time.round(4))

,option,rows,auc,log_loss
0,"type of work, customer source, sending speed (...",5990,0.6388,0.6137
1,"+ industry, customer type, quote size, number ...",5990,0.6373,0.6153
2,+ the business's past win rate,5990,0.6386,0.6136
3,+ the customer's past win rate,5990,0.6355,0.6164
4,+ sent on a weekend,5990,0.6386,0.6136
5,without type of work,5990,0.5812,0.6319
6,without customer source,5990,0.6234,0.6185
7,without sending speed (quote size instead),5990,0.6295,0.6165
8,gradient boosting with every fact,5990,0.5867,0.7086


Three facts do the work: the type of work, how the customer found the business, and how fast the
quote went out. Adding industry, customer type, quote size, number of items, weekend sending, or
either history rate adds nothing, which matches the finding in notebook 02 that businesses don't
differ beyond their mix. Taking any one of the three out makes things worse, the type of work
most of all.

A more complex model (gradient boosting) does clearly worse. With only a few real patterns to
find, its extra flexibility mostly picks up noise, and it couldn't explain its scores the way
the simple model can.

## How best to use "days without a reply"?

Each option is scored at every point a quote was still waiting (after 0, 1, 2, 3, 5, 7, 10, 14
and 21 days). The last columns are the H3 check: how many groups of similar quotes the chances
missed by more than luck explains, against the threshold shown.

In [3]:
silence = ablations.silence_ablations(quotes, settings)
display(silence.round(4))

,option,rows,auc,log_loss,groups_missed_beyond_luck,worst_miss_z,luck_threshold_z
0,one says-no-or-goes-quiet figure per type of w...,24151,0.7599,0.5717,3,4.6545,3.4101
1,says-no-or-goes-quiet chance per customer (cho...,24151,0.7616,0.5705,1,3.6614,3.4101
2,"same, reply speed also learned by customer s...",24151,0.7612,0.5711,2,4.7092,3.4101
3,"same, reply speed also learned by sending speed",24151,0.7619,0.5704,3,4.7393,3.4101
4,"same, with gradient boosting instead",24151,0.7336,0.6137,7,4.8450,3.4101
5,days waiting as one more fact,24151,0.7592,0.5733,4,4.1777,3.4101
6,"days waiting as one more fact, gradient boosting",24151,0.7372,0.6257,3,3.8465,3.4101
7,days waiting only,24151,0.7190,0.5920,27,9.5049,3.4101


What this shows:

- **Letting "says no vs goes quiet" depend on the customer is the fix that matters.** My first
  version used one figure per type of work and missed in 3 groups. Letting it vary by customer,
  as notebook 03 suggests, scores better (0.5705 against 0.5717) and misses in just 1.
- **Learning reply speed separately for smaller groups doesn't help.** Splitting by sending speed
  ties on log loss (0.5704), and splitting by customer source is slightly worse (0.5711). Both miss
  in more groups (3 and 2), so reply speed stays learned per type of work.
- **The approach beats the usual alternative**, which simply adds "days waiting" as one more fact
  in the model (van Houwelingen 2007). It also keeps "days without a reply" as its own reason on
  the board.
- **Gradient boosting does worse here too.**
- **Days waiting on its own is badly off for many groups** (27 misses). That matters for H2 and H3.

So the model going forward is the one in `config.toml`: the three facts, a "says no vs goes quiet"
chance for each customer, and reply speeds learned per type of work.

In [4]:
print((ROOT / "config.toml").read_text().split("[followup]")[0])

# Every number that shapes a result lives here, so a run is fully described by this file and the data.

[data]
dir = "data"

[model]
# Chosen in notebooks/04 on folds that close before the backtest: industry, amount, line items,
# customer type and SP or customer history added nothing once these three were in. The same
# inputs feed both the win model and the decline-share model.
categorical = ["workflow_type", "customer_channel"]
numeric = ["log_hours_to_send"]
regularization_c = 1.0
# Train only on quotes sent at least this long before the scoring date. Quotes expire after
# 30 days and almost every decision lands inside 45, so their outcomes are settled and recent
# fast "yes" answers can't crowd out the slow "no" ones still pending.
label_horizon_days = 45
# Strata for the answer-time curves, coarsest first. Thin strata back off to coarser ones.
timing_segments = ["workflow_type"]
timing_min_quotes = 200


